# Student Anxiety Risk Prediction Pipeline
## MindNova Analytics Module

This notebook orchestrates the end-to-end machine learning pipeline to predict student mental health risk.

### Pipeline Overview:
1. **Data Loading & EDA**: Understanding the dataset and target distribution.
2. **Data Cleaning**: Handling duplicates, missing values, and outliers.
3. **Feature Engineering**: Creating composite risk and wellness scores.
4. **Train-Test-Validation Split**: Stratified 70/15/15 split.
5. **Handling Imbalance**: SMOTE for minority class oversampling.
6. **Model Benchmarking**: Version A (Diagnostic) vs Version B (Behavioral/Production).
7. **Evaluation & Tuning**: RandomizedSearchCV for peak Recall.
8. **Explainability**: SHAP and LIME for clinical insights.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Append src to path for modular imports
sys.path.append(os.path.abspath('../src'))

import anxiety_preprocess as preprocess
import anxiety_feature_engineering as engineering
import anxiety_train as training
import anxiety_evaluate as evaluate
import anxiety_tune as tuning
import anxiety_explain as explain

## 1. Data Loading & Initial Inspection

In [ ]:
DATA_PATH = '../data/raw/Univsersiyt_Student_Mental_health_data.csv'
df = preprocess.load_data(DATA_PATH)
preprocess.basic_inspection(df)

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Target Class Distribution Plot
plt.figure(figsize=(6, 4))
sns.countplot(x='MentalHealthStatus', data=df, palette='viridis')
plt.title('Target Distribution (0: Low Risk, 1: High Risk)')
plt.show()

# Correlation Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Feature Correlation Heatmap')
plt.show()

## 3. Preprocessing & Feature Engineering

In [ ]:
# Clean data
df_clean = preprocess.clean_data(df)

# Feature Engineering
df_feat = engineering.engineer_features(df_clean)

# Create Versions
(X_A, y_A), (X_B, y_B) = engineering.get_feature_versions(df_feat)

print(f"Version A columns: {len(X_A.columns)}")
print(f"Version B columns: {len(X_B.columns)} (Diagnostic features removed)")

## 4. Train-Test Split & SMOTE Balancing

In [ ]:
from sklearn.model_selection import train_test_split

# Using Version B for the rest of the production pipeline
X_train, X_temp, y_train, y_temp = train_test_split(X_B, y_B, test_size=0.3, random_state=42, stratify=y_B)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# Handle Imbalance on training set
X_train_res, y_train_res = training.solve_imbalance(X_train, y_train)

print(f"Balanced Training samples: {len(X_train_res)}")

## 5. Model Benchmarking & Selection

In [ ]:
# Train baselines
base_models = training.train_baseline_models(X_train_res, y_train_res)

# Train NN (5 layers)
input_dim = X_train_res.shape[1]
# nn_model = training.train_neural_network(X_train_res, y_train_res, input_dim)
# base_models["Neural Network"] = nn_model

# Evaluate all
all_metrics = []
for name, model in base_models.items():
    metrics, _, _ = evaluate.evaluate_model(name, model, X_val, y_val)
    all_metrics.append(metrics)

summary = evaluate.get_summary_table(all_metrics)
display(summary)

## 6. Hyperparameter Tuning

In [ ]:
# Tuning the top performer (e.g., XGBoost)
best_xgb, best_params = tuning.tune_xgboost(X_train_res, y_train_res)
print(f"Best XGBoost Params: {best_params}")

# Final Evaluation on Test Set
test_metrics, y_pred, y_prob = evaluate.evaluate_model("Tuned XGBoost (Final)", best_xgb, X_test, y_test)
print("\n--- Final Test Metrics ---")
display(pd.DataFrame([test_metrics]))

evaluate.plot_confusion_matrix(y_test, y_pred, "Final Model")

## 7. Model Explainability

In [ ]:
# Generate SHAP
# explain.generate_shap_plots(best_xgb, X_train_res, X_test, X_B.columns, '../figures/final_model')

# Generate LIME for high-risk sample
sample_idx = 0
exp = explain.generate_lime_explanation(best_xgb, X_train_res, X_test, X_B.columns, sample_idx)
# exp.show_in_notebook(show_table=True)